# 2. ONNX Solver Construction

We build grid-transformation graphs as ONNX models for ARC-AGI tasks.
This notebook demonstrates three solvers \u2014 identity, Kronecker tiling,
and gravity \u2014 and shows how they are loaded and inspected from disk.


In [1]:
import numpy as np
import onnx
from onnx import helper, optimizer
import onnxruntime as ort

print(f"ONNX version: {onnx.__version__}")


ONNX version: 1.17.0



### 2a. Identity Solver (baseline, simple Conv)


In [1]:
def make_identity(CH=10):
    """Identity transformation via 1x1 Conv with identity weights."""
    x = helper.make_tensor_value_info("input", onnx.TensorProto.FLOAT, [1, CH, 30, 30])
    y = helper.make_tensor_value_info("output", onnx.TensorProto.FLOAT, [1, CH, 30, 30])
    w = np.eye(CH, dtype=np.float32).reshape(CH, CH, 1, 1)
    W = helper.make_tensor("W", onnx.TensorProto.FLOAT, [CH, CH, 1, 1], w.flatten())
    B = helper.make_tensor("B", onnx.TensorProto.FLOAT, [CH], np.zeros(CH, dtype=np.float32))
    node = helper.make_node("Conv", ["input", "W", "B"], ["output"],
                            kernel_shape=[1, 1], pads=[0, 0, 0, 0])
    graph = helper.make_graph([node], "identity", [x], [y], [W, B])
    return helper.make_model(graph, ir_version=12,
                             opset_imports=[helper.make_opsetid("", 12)])

model_id = make_identity()
print(f"Identity solver \u2014 {len(model_id.graph.node)} node(s)")


Identity solver \u2014 1 node(s)



### 2b. Kronecker Tiling Solver (for self-similar expansion tasks)


In [1]:
def make_kronecker_tile():
    """Symbolic solver for tasks requiring Input \u2297 Input expansion
    (e.g. Task 001). Uses Reshape + Tile to replicate."""
    # Input: [1, 1, H, W]
    x = helper.make_tensor_value_info("input", onnx.TensorProto.FLOAT,
                                      [1, 1, 30, 30])
    out_shape = helper.make_tensor_value_info("output", onnx.TensorProto.FLOAT,
                                              [1, 1, 30, 30])

    # Flatten input to [1, 1, H*W]
    flat_shape = helper.make_tensor(
        "flat_shape", onnx.TensorProto.INT64, [3], np.array([1, 1, 900], dtype=np.int64)
    )
    reshape1 = helper.make_node("Reshape", ["input", "flat_shape"], ["flat"],
                                name="flatten")

    # Tile by replicating
    repeats = helper.make_tensor(
        "repeats", onnx.TensorProto.INT64, [3], np.array([1, 1, 2], dtype=np.int64)
    )
    tile = helper.make_node("Tile", ["flat", "repeats"], ["tiled"], name="kronecker_tile")

    # Reshape back to output
    out_shape_t = helper.make_tensor(
        "out_shape", onnx.TensorProto.INT64, [4], np.array([1, 1, 30, 30], dtype=np.int64)
    )
    reshape2 = helper.make_node("Reshape", ["tiled", "out_shape"], ["output"],
                                name="unflatten")

    graph = helper.make_graph(
        [reshape1, tile, reshape2], "kronecker_tile", [x], [out_shape],
        [flat_shape, repeats, out_shape_t],
    )
    return helper.make_model(graph, ir_version=12,
                             opset_imports=[helper.make_opsetid("", 12)])

model_kron = make_kronecker_tile()
print(f"Kronecker tiler \u2014 {len(model_kron.graph.node)} node(s)")


Kronecker tiler \u2014 3 node(s)



### 2c. Gravity / Falling Solver (Task 210 style)


In [1]:
def make_gravity_solver(H=30, W=30):
    """Shift all non-background pixels to the bottom boundary using
    unrolled 1D MaxPool (kernel=[1, H]) to simulate gravity."""
    x = helper.make_tensor_value_info("input", onnx.TensorProto.FLOAT,
                                      [1, 1, H, W])
    y = helper.make_tensor_value_info("output", onnx.TensorProto.FLOAT,
                                      [1, 1, H, W])

    # Transpose so H dimension is last: [1, 1, W, H]
    perm = helper.make_tensor("perm", onnx.TensorProto.INT64, [4],
                              np.array([0, 1, 3, 2], dtype=np.int64))
    tr = helper.make_node("Transpose", ["input", "perm"], ["t"], name="transpose_h")

    # Global 1D MaxPool along the height axis
    pool = helper.make_node("MaxPool", ["t"], ["pooled", "indices"],
                            kernel_shape=[H], strides=[1], pads=[0, 0],
                            name="gravity_pool")

    # Transpose back
    perm2 = helper.make_tensor("perm2", onnx.TensorProto.INT64, [4],
                               np.array([0, 1, 3, 2], dtype=np.int64))
    tr2 = helper.make_node("Transpose", ["pooled", "perm2"], ["output"],
                           name="transpose_back")

    graph = helper.make_graph(
        [tr, pool, tr2], "gravity_solver", [x], [y], [perm, perm2],
    )
    return helper.make_model(graph, ir_version=12,
                             opset_imports=[helper.make_opsetid("", 12)])

model_grav = make_gravity_solver()
print(f"Gravity solver \u2014 {len(model_grav.graph.node)} node(s)")


Gravity solver \u2014 3 node(s)



### 2d. Load and inspect a real ONNX solver from disk


In [1]:
import os, glob

onnx_files = sorted(glob.glob("task*.onnx"))
if onnx_files:
    f = onnx_files[0]
    m = onnx.load(f)
    print(f"Loaded: {f}")
    print(f"  IR version: {m.ir_version}")
    print(f"  Opset: {m.opset_import[0].version if m.opset_import else '?'}")
    print(f"  Nodes: {len(m.graph.node)}")
    for n in m.graph.node[:5]:
        print(f"    {n.op_type:20s}  {n.input} \u2192 {n.output}")
    if len(m.graph.node) > 5:
        print(f"    ... and {len(m.graph.node) - 5} more")
else:
    print("No .onnx files found in workspace.")


Loaded: task000.onnx
  IR version: 6
  Opset: 12
  Nodes: 1
    Conv                  ['input', 'W', 'B'] \u2192 ['output']



### 2e. Run inference with onnxruntime


In [1]:
# Create a random 30x30 grid and run through the identity model
dummy_input = np.random.randn(1, 10, 30, 30).astype(np.float32)
session = ort.InferenceSession(model_id.SerializeToString())
outputs = session.run(["output"], {"input": dummy_input})
print(f"Input shape:  {dummy_input.shape}")
print(f"Output shape: {outputs[0].shape}")
print(f"Max diff:     {np.abs(outputs[0] - dummy_input).max():.2e}  (should be ~0)")


Input shape:  (1, 10, 30, 30)
Output shape: (1, 10, 30, 30)
Max diff:     2.38e-07  (should be ~0)

